In [7]:
# Cell 1: Imports & Load Data
import json
from pathlib import Path
from collections import Counter

PROJECT = Path.home() / "llm-mail-trainer"

# Load parsed emails (40k)
with open(PROJECT / "data/parsed/emails.json", 'r') as f:
    parsed_emails = json.load(f)

# Load classification results (499)
with open(PROJECT / "data/parsed/classification_results.json", 'r') as f:
    results = json.load(f)

print(f"✅ Loaded {len(parsed_emails):,} parsed emails")
print(f"✅ Loaded {len(results)} classification results")

# Cell 2: Extract finance emails with full details
finance_results = [r for r in results if r['category'] == 'finance']

# Get full email data for finance emails
finance_ids = [r['id'] for r in finance_results]
finance_emails_full = [e for e in parsed_emails if e['id'] in finance_ids]

print(f"Finance emails found: {len(finance_emails_full)}")

# Show senders
print("\n=== TOP FINANCE SENDERS ===")
senders = [e['sender'] for e in finance_emails_full]
for sender, count in Counter(senders).most_common(15):
    print(f"  {count:3} : {sender[:60]}")

# Cell 3: Check emails with empty sender
empty_sender_emails = [e for e in finance_emails_full if e['sender'].strip() == '']

print(f"Emails with empty sender: {len(empty_sender_emails)}\n")

for i, email in enumerate(empty_sender_emails[:5]):
    print(f"{i+1}. Subject: {email['subject'][:70]}")
    print(f"   Body: {email['body'][:150]}...")
    print()

# Cell 4: View finance email bodies (for pattern discovery)
print("=== SAMPLE FINANCE EMAIL BODIES ===\n")

# Filter only emails with known finance senders
finance_senders = ['icici', 'hdfc', 'groww', 'zerodha', 'paisabazaar', 'sbi', 'axis', 'kotak']

real_finance = []
for e in finance_emails_full:
    sender_lower = e['sender'].lower()
    subject_lower = e['subject'].lower()
    body_lower = e['body'].lower()
    
    # Check sender or content
    if any(fs in sender_lower for fs in finance_senders):
        real_finance.append(e)
    elif any(kw in body_lower for kw in ['debited', 'credited', 'transaction', 'upi', 'a/c', 'balance']):
        real_finance.append(e)

print(f"Real finance emails: {len(real_finance)}\n")

# Show 5 samples
for i, email in enumerate(real_finance[:5]):
    print(f"--- Email {i+1} ---")
    print(f"Subject: {email['subject'][:80]}")
    print(f"Sender: {email['sender']}")
    print(f"Body: {email['body'][:300]}")
    print()

# Cell 5: Pattern extraction functions
import re

def extract_entities(text):
    """Extract financial entities from email text."""
    
    entities = {}
    
    # Amount: Rs.1890.28 or Rs 1,890.28 or ₹1890
    amount_match = re.search(r'(?:Rs\.?|₹)\s*([\d,]+(?:\.\d{2})?)', text)
    if amount_match:
        entities['amount'] = amount_match.group(1).replace(',', '')
    
    # Type: debited or credited
    if 'debited' in text.lower():
        entities['type'] = 'debit'
    elif 'credited' in text.lower():
        entities['type'] = 'credit'
    
    # Account: account XXXX or A/C XXXX
    account_match = re.search(r'(?:account|A/C|a/c)\s*[:\s]?\s*(\w+)', text, re.IGNORECASE)
    if account_match:
        entities['account'] = account_match.group(1)
    
    # Date: DD-MM-YY or DD-MM-YYYY
    date_match = re.search(r'(\d{2}-\d{2}-\d{2,4})', text)
    if date_match:
        entities['date'] = date_match.group(1)
    
    # UPI Reference
    ref_match = re.search(r'reference\s*(?:number|no\.?)?\s*(?:is)?\s*(\d+)', text, re.IGNORECASE)
    if ref_match:
        entities['reference'] = ref_match.group(1)
    
    return entities

# Test on sample
sample_text = real_finance[0]['body']
print("=== INPUT TEXT ===")
print(sample_text[:300])
print("\n=== EXTRACTED ENTITIES ===")
extracted = extract_entities(sample_text)
for key, value in extracted.items():
    print(f"  {key}: {value}")

# Cell 6: Test on transaction emails
print("=== TESTING ENTITY EXTRACTION ===\n")

# Find emails with "debited" or "credited"
transaction_emails = [e for e in real_finance if 'debited' in e['body'].lower() or 'credited' in e['body'].lower()]

print(f"Transaction emails found: {len(transaction_emails)}\n")

for i, email in enumerate(transaction_emails[:3]):
    print(f"--- Email {i+1} ---")
    print(f"Subject: {email['subject'][:60]}")
    print(f"Body: {email['body'][:200]}...")
    print(f"\nExtracted:")
    extracted = extract_entities(email['body'])
    for key, value in extracted.items():
        print(f"  {key}: {value}")
    print()

# Cell 7: Save patterns and filtered finance emails

# Create filtered directory
filtered_dir = PROJECT / "data/filtered"
filtered_dir.mkdir(exist_ok=True)

# Save real finance emails
with open(filtered_dir / "finance_emails.json", 'w') as f:
    json.dump(real_finance, f, ensure_ascii=False, indent=2)

# Save patterns configuration
patterns = {
    "amount": r"(?:Rs\.?|₹)\s*([\d,]+(?:\.\d{2})?)",
    "type_debit": "debited",
    "type_credit": "credited",
    "account": r"(?:account|A/C|a/c)\s*[:\s]?\s*(\w+)",
    "date": r"(\d{2}-\d{2}-\d{2,4})",
    "reference": r"reference\s*(?:number|no\.?)?\s*(?:is)?\s*(\d+)",
    "finance_senders": ["icici", "hdfc", "groww", "zerodha", "paisabazaar", "sbi", "axis", "kotak"],
    "finance_keywords": ["debited", "credited", "transaction", "upi", "a/c", "balance", "payment"]
}

with open(filtered_dir / "finance_patterns.json", 'w') as f:
    json.dump(patterns, f, indent=2)

print(f"✅ Saved {len(real_finance)} finance emails to data/filtered/finance_emails.json")
print(f"✅ Saved patterns to data/filtered/finance_patterns.json")

✅ Loaded 40,820 parsed emails
✅ Loaded 499 classification results
Finance emails found: 89

=== TOP FINANCE SENDERS ===
   18 : 
    7 : "ICICI Bank Privilege Banking"
    5 : ICICI Bank
    5 : "HDFC Life"
    4 : HDFC Bank InstaAlerts
    4 : Groww Digest
    3 : Quora Digest
    3 : Groww
    3 : HDFC Life - no reply
    2 : "alert @ icicibank . com"
    2 : Paisabazaar Support
    2 : LinkedIn
    2 : Google Play
    2 : "ICICI Bank "
    1 : "Zerodha Broking Ltd"
Emails with empty sender: 18

1. Subject: Funds/Securities Balance
   Body: Dear Investor, With reference to NSE circulars NSE/INSP/46704 dated December 17, 2020, and NSE/INSP/55039 dated December 28, 2022, Trading members are...

2. Subject: Dear Customer, Rs.120.00 has been debited from account 1234 to VPA com
   Body: Dear Customer, Rs.120.00 has been debited from account 1234 to VPA companyonline@ybl pepto on 06-12-25. Your UPI transaction reference number is 50000...

3. Subject: 40GB of Data allocated to Jio Number 